# Student-Friendly Fixed Notebook

This version is written for MSc students who may be new to programming and artificial intelligence. It uses the same U-Net segmentation workflow, but the code contains extra comments explaining what each important line is doing.

Important runtime setting: this notebook uses `NUM_WORKERS = 0` and `PIN_MEMORY = False`. These settings avoid the PyTorch `DataLoader` multiprocessing cleanup error that can appear in Jupyter notebooks.


# Liver and Tumor Segmentation with U-Net in PyTorch

This notebook trains a **2D U-Net** model to perform medical image segmentation on CT slices. Segmentation means assigning a class label to every pixel in an image.

## Dataset used in this practical

For this practical, we used the **HCC subset** of a public primary liver cancer contrast-enhanced CT dataset. HCC means **hepatocellular carcinoma**, the most common type of primary liver cancer.

The source dataset is described in the Scientific Data article **"Comprehensive multi-phase 3D contrast-enhanced CT imaging for primary liver cancer"**: https://www.nature.com/articles/s41597-025-05125-2

The full dataset contains multiple primary liver cancer subtypes and CT phases. In this notebook, we focus on the **arterial phase HCC data from 94 patients**. The data have been prepared as 2D PNG slices with matching segmentation masks so that students can train a 2D U-Net without needing to handle the original 3D NIFTI files directly.

In this practical, each pixel belongs to one of three classes:

- `0`: background, shown as black in the mask
- `1`: liver, shown as green in the mask
- `2`: tumor, shown as red in the mask

A CT image is the input. A mask is the answer we want the model to learn. During training, the model sees both the image and the correct mask. Later, it receives only an image and predicts a mask.

The data folder used by this notebook is:

```text
Segmentation/
  train/
    images/
    masks/
  val/
    images/
    masks/
  test/
    images/
    masks/
```

Why three splits?

- `train`: used to update the model weights.
- `val`: used during training to check whether the model is improving.
- `test`: used at the end for the final report.

The workflow is:

1. Check the data.
2. Convert color masks into class numbers.
3. Build PyTorch datasets and dataloaders.
4. Define a U-Net model.
5. Train the model.
6. Evaluate Dice and IoU scores.
7. Display and save predictions.

> Medical note: this notebook is for education and research practice only. It is not a clinical tool.


## 1. Install and import packages

A package is a collection of code written by other people. We use packages so we do not need to implement everything from scratch.

The most important packages here are:

- `torch`: PyTorch, the deep learning library.
- `PIL`: opens PNG images.
- `numpy`: handles arrays, which are tables of numbers.
- `matplotlib`: shows images and plots.
- `tqdm`: shows progress bars.
- `pandas`: displays training results as tables.

Run the next cell first. If a package is missing, install it in the notebook environment, then restart the kernel and run again.


In [ ]:
# If needed, uncomment and run this line once.
# The exclamation mark means: run this as a terminal command from inside Jupyter.
# After installing packages, restart the kernel so Python can find them correctly.
# !pip install torch torchvision pillow matplotlib numpy pandas tqdm scikit-learn

# pathlib gives a clean way to work with file and folder paths.
from pathlib import Path

# random is used for random image selection and random data augmentation.
import random

# time is used to measure how long training takes.
import time

# gc is Python's garbage collector. We use it to clean old DataLoader objects in notebooks.
import gc

# NumPy stores image data as arrays of numbers.
import numpy as np

# pandas displays results in tables.
import pandas as pd

# PIL opens image files. ImageOps gives simple image transformations such as flips.
from PIL import Image, ImageOps

# matplotlib displays images and training curves.
import matplotlib.pyplot as plt

# tqdm shows progress bars. This plain version avoids Jupyter widget version errors.
from tqdm import tqdm

# PyTorch imports.
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Print basic environment information so students know what is being used.
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


## 2. Configuration

This cell stores values that control the whole experiment. Keeping them in one place makes the notebook easier to change.

Key ideas:

- `IMAGE_SIZE`: all images are resized to this square size.
- `BATCH_SIZE`: how many images the model sees before one weight update.
- `EPOCHS`: how many full passes through the training set.
- `LEARNING_RATE`: how large each optimizer update is.
- `NUM_CLASSES`: three classes: background, liver, tumor.
- `DEVICE`: uses GPU with CUDA if available, otherwise CPU.

For a first classroom run, use `IMAGE_SIZE = 256` and `EPOCHS = 5`. For a stronger final model, try `IMAGE_SIZE = 512` and more epochs if a GPU is available.


In [ ]:
# Reproducibility: this helps us get similar results when we rerun the notebook.
# Some randomness still exists on GPUs, but setting seeds makes runs more comparable.
SEED = 42
random.seed(SEED)          # seed for Python's random module
np.random.seed(SEED)       # seed for NumPy random numbers
torch.manual_seed(SEED)    # seed for PyTorch on CPU
torch.cuda.manual_seed_all(SEED)  # seed for PyTorch on GPU, if available

# Data location. Path('.') means the current folder where this notebook is saved.
DATA_DIR = Path('.')
TRAIN_DIR = DATA_DIR / 'train'
VAL_DIR = DATA_DIR / 'val'
TEST_DIR = DATA_DIR / 'test'

# Training settings.
IMAGE_SIZE = 256       # All images are resized to 256 x 256 pixels.
BATCH_SIZE = 8         # The model processes 8 images before each weight update.
EPOCHS = 5             # One epoch means one full pass through the training dataset.
LEARNING_RATE = 1e-3   # Step size used by the optimizer. 1e-3 means 0.001.
NUM_CLASSES = 3        # background, liver, tumor

# Jupyter-safe DataLoader settings.
# Keeping workers at 0 avoids multiprocessing cleanup errors in notebooks.
NUM_WORKERS = 0
PIN_MEMORY = False

# Use a GPU if PyTorch can see one. Otherwise use the CPU.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)
print('DataLoader workers:', NUM_WORKERS, '| pin_memory:', PIN_MEMORY)


## 3. Check the dataset

Before training, always check that every image has a matching mask.

The image and mask filenames should be identical. For example:

```text
train/images/P0001_C1_slice_010.png
train/masks/P0001_C1_slice_010.png
```

If an image has no mask, the model cannot learn the correct answer for that image. If a mask has no image, the notebook cannot load it as a training example.


In [ ]:
def list_pngs(folder):
    """Return a sorted list of PNG files in one folder."""
    # sorted(...) gives a stable order, which makes results easier to reproduce.
    return sorted(Path(folder).glob('*.png'))

# Check train, validation, and test folders one by one.
for split in ['train', 'val', 'test']:
    image_paths = list_pngs(DATA_DIR / split / 'images')
    mask_paths = list_pngs(DATA_DIR / split / 'masks')
    print(f'{split:5s}: {len(image_paths):5d} images, {len(mask_paths):5d} masks')

    # We compare filenames, not full paths, because images and masks are in different folders.
    image_names = {p.name for p in image_paths}
    mask_names = {p.name for p in mask_paths}

    # Set subtraction finds files that exist on one side but not the other.
    missing_masks = image_names - mask_names
    missing_images = mask_names - image_names

    if missing_masks or missing_images:
        print('  Problem found: image/mask names do not match.')
        print('  Missing masks:', sorted(list(missing_masks))[:5])
        print('  Missing images:', sorted(list(missing_images))[:5])
    else:
        print('  OK: all images have matching masks.')


## 4. Understand the mask colors

The mask PNG files are color images, but neural networks train on numbers, not color names.

This notebook converts colors into class numbers:

- black `(0, 0, 0)` becomes class `0` = background
- green `(0, 255, 0)` becomes class `1` = liver
- red `(255, 0, 0)` becomes class `2` = tumor

The model will later output three scores for every pixel. The class with the highest score becomes the predicted class for that pixel.


In [ ]:
# These class names define the meaning of each class number.
# The order matters: class 0 is background, class 1 is liver, class 2 is tumor.
CLASS_NAMES = ['background', 'liver', 'tumor']

# RGB colors used when displaying or saving masks.
# dtype=np.uint8 means each color channel is an integer from 0 to 255.
CLASS_COLORS = np.array([
    [0, 0, 0],       # class 0: background: black
    [0, 255, 0],     # class 1: liver: green
    [255, 0, 0],     # class 2: tumor: red
], dtype=np.uint8)


def rgb_mask_to_class(mask_rgb):
    """Convert an RGB mask image into a 2D class-index mask.

    Input shape:  H x W x 3 with colors black/green/red.
    Output shape: H x W with values 0, 1, or 2.
    """
    # Convert the PIL image into a NumPy array so we can compare pixel values.
    mask_rgb = np.asarray(mask_rgb, dtype=np.uint8)

    # Start with all pixels as background class 0.
    class_mask = np.zeros(mask_rgb.shape[:2], dtype=np.int64)

    # np.all(..., axis=-1) checks all three RGB channels for each pixel.
    liver_pixels = np.all(mask_rgb == [0, 255, 0], axis=-1)
    tumor_pixels = np.all(mask_rgb == [255, 0, 0], axis=-1)

    # Replace selected pixels with their class numbers.
    class_mask[liver_pixels] = 1
    class_mask[tumor_pixels] = 2
    return class_mask


def class_mask_to_rgb(class_mask):
    """Convert class numbers back to RGB colors for display or saving."""
    # Example: class value 2 becomes CLASS_COLORS[2], which is red.
    return CLASS_COLORS[np.asarray(class_mask, dtype=np.int64)]

# Check a few masks to confirm they contain expected class values.
for mask_path in list_pngs(TRAIN_DIR / 'masks')[:3]:
    mask_rgb = Image.open(mask_path).convert('RGB')
    class_mask = rgb_mask_to_class(mask_rgb)
    print(mask_path.name, 'unique class values:', np.unique(class_mask))


## 5. Visualize images and masks

It is important to look at the data. This helps catch mistakes such as masks being shifted, wrong colors, or blank labels.


In [ ]:
def show_examples(split='train', n=3):
    """Display CT images, true masks, and overlays for a few examples."""
    image_paths = list_pngs(DATA_DIR / split / 'images')

    # Pick a small random set of examples. min(...) avoids asking for more images than exist.
    chosen = random.sample(image_paths, k=min(n, len(image_paths)))

    # We create 3 columns: image, mask, and overlay.
    fig, axes = plt.subplots(len(chosen), 3, figsize=(12, 4 * len(chosen)))
    if len(chosen) == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, image_path in enumerate(chosen):
        mask_path = DATA_DIR / split / 'masks' / image_path.name

        # CT images are displayed as grayscale. Masks are read as RGB colors.
        image = Image.open(image_path).convert('L')
        mask_rgb = Image.open(mask_path).convert('RGB')
        class_mask = rgb_mask_to_class(mask_rgb)

        axes[row, 0].imshow(image, cmap='gray')
        axes[row, 0].set_title('CT image')
        axes[row, 0].axis('off')

        axes[row, 1].imshow(class_mask_to_rgb(class_mask))
        axes[row, 1].set_title('Ground-truth mask')
        axes[row, 1].axis('off')

        # alpha controls transparency. Lower alpha means the CT image remains visible.
        axes[row, 2].imshow(image, cmap='gray')
        axes[row, 2].imshow(class_mask_to_rgb(class_mask), alpha=0.35)
        axes[row, 2].set_title('Overlay')
        axes[row, 2].axis('off')

    plt.tight_layout()
    plt.show()

show_examples('train', n=3)


## 6. PyTorch Dataset

A PyTorch `Dataset` is a small class that tells PyTorch how to load one example.

One example means:

- one CT image
- the matching segmentation mask

For each example we will:

1. Open the CT image as grayscale. A grayscale image has one channel.
2. Open the mask as RGB. RGB means red, green, and blue channels.
3. Resize both to the same size.
4. Convert the RGB mask into class numbers.
5. Convert everything into PyTorch tensors.

A tensor is PyTorch's version of an array. The model expects tensors, not PIL images or NumPy arrays.


In [ ]:
class LiverTumorDataset(Dataset):
    """Dataset for paired CT image and segmentation mask PNG files."""

    def __init__(self, root_dir, image_size=256, augment=False):
        # root_dir is one split folder, for example train, val, or test.
        self.root_dir = Path(root_dir)
        self.image_dir = self.root_dir / 'images'
        self.mask_dir = self.root_dir / 'masks'

        # Store all image paths. Each mask is found later by using the same filename.
        self.image_paths = list_pngs(self.image_dir)
        self.image_size = image_size
        self.augment = augment

        if len(self.image_paths) == 0:
            raise ValueError(f'No PNG images found in {self.image_dir}')

    def __len__(self):
        # PyTorch calls this to know how many examples are in the dataset.
        return len(self.image_paths)

    def __getitem__(self, idx):
        # PyTorch calls this to load one example by index.
        image_path = self.image_paths[idx]
        mask_path = self.mask_dir / image_path.name

        # The CT image is grayscale, so it has one channel.
        image = Image.open(image_path).convert('L')

        # The mask uses RGB colors because each color represents a class.
        mask = Image.open(mask_path).convert('RGB')

        # Resize image with bilinear interpolation for smooth intensity values.
        # Resize mask with nearest-neighbor so class colors do not get blurred.
        image = image.resize((self.image_size, self.image_size), resample=Image.BILINEAR)
        mask = mask.resize((self.image_size, self.image_size), resample=Image.NEAREST)

        if self.augment:
            # Data augmentation creates slightly different training examples.
            # The same flip must be applied to image and mask so they stay aligned.
            if random.random() < 0.5:
                image = ImageOps.mirror(image)
                mask = ImageOps.mirror(mask)
            if random.random() < 0.5:
                image = ImageOps.flip(image)
                mask = ImageOps.flip(mask)

        # Convert image pixels from 0..255 integers to 0..1 floating-point values.
        image_array = np.asarray(image, dtype=np.float32) / 255.0

        # Convert colored mask pixels into class numbers 0, 1, 2.
        mask_array = rgb_mask_to_class(mask)

        # PyTorch Conv2d expects image shape: channels x height x width.
        # The grayscale image starts as height x width, so unsqueeze(0) adds one channel.
        image_tensor = torch.from_numpy(image_array).unsqueeze(0)

        # Cross-entropy loss expects class labels as integer type long.
        mask_tensor = torch.from_numpy(mask_array).long()

        return image_tensor, mask_tensor


# Create one Dataset object for each split.
train_dataset = LiverTumorDataset(TRAIN_DIR, image_size=IMAGE_SIZE, augment=True)
val_dataset = LiverTumorDataset(VAL_DIR, image_size=IMAGE_SIZE, augment=False)
test_dataset = LiverTumorDataset(TEST_DIR, image_size=IMAGE_SIZE, augment=False)

print('Train samples:', len(train_dataset))
print('Validation samples:', len(val_dataset))
print('Test samples:', len(test_dataset))

# Load one example to check shapes before training.
image_tensor, mask_tensor = train_dataset[0]
print('One image tensor shape:', image_tensor.shape)
print('One mask tensor shape:', mask_tensor.shape)
print('Mask classes:', torch.unique(mask_tensor))


In [ ]:
# Remove any old DataLoader objects from earlier notebook runs.
# This is useful in Jupyter because variables can remain alive after re-running cells.
for loader_name in ('train_loader', 'val_loader', 'test_loader'):
    if loader_name in globals():
        del globals()[loader_name]
gc.collect()

# A DataLoader groups individual examples into batches.
# The training loader shuffles data so the model does not see images in the same order every epoch.
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

# Validation and test loaders do not need shuffling because we only measure performance.
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print('DataLoader workers:', NUM_WORKERS, '| pin_memory:', PIN_MEMORY)

# Inspect one batch. Batch image shape is: batch x channels x height x width.
batch_images, batch_masks = next(iter(train_loader))
print('Batch image shape:', batch_images.shape)
print('Batch mask shape:', batch_masks.shape)


## 7. U-Net model

U-Net is popular for medical image segmentation because it keeps both global context and fine detail.

The architecture has three main parts:

- Encoder / downsampling path: looks at larger image context.
- Bottleneck: the deepest, most compressed representation.
- Decoder / upsampling path: returns to the original image size.

In this notebook, the U-Net uses `features=(32, 64, 128, 256)`. This means:

- **4 encoder stages**: 32, 64, 128, and 256 feature channels.
- **1 bottleneck stage**: 512 feature channels.
- **4 decoder stages**: 256, 128, 64, and 32 feature channels.
- **4 skip connections**: one from each encoder stage to the matching decoder stage.

Each encoder stage uses a `DoubleConv` block. A `DoubleConv` block contains two convolution layers. Therefore, the encoder has **8 convolution layers** in total.

The bottleneck also uses one `DoubleConv` block, so it has **2 convolution layers**.

Each decoder stage has one upsampling layer (`ConvTranspose2d`) and one `DoubleConv` block. Therefore, the decoder has **4 upsampling layers** and **8 convolution layers**.

At the end, a final `1 x 1` convolution converts the last 32 feature channels into 3 class-score channels: background, liver, and tumor.

Skip connections copy feature maps from the encoder to the decoder. This helps the model recover edges and small structures, which is important for tumors.

The model output has shape `batch x 3 x height x width`. The three channels are raw scores for background, liver, and tumor. These raw scores are called `logits`.


In [ ]:
class DoubleConv(nn.Module):
    """Two convolution layers used repeatedly inside U-Net."""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            # Conv2d learns small filters that detect local patterns such as edges and textures.
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            # BatchNorm helps training remain stable.
            nn.BatchNorm2d(out_channels),
            # ReLU adds non-linearity so the model can learn complex patterns.
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=3, features=(32, 64, 128, 256)):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Encoder: each step learns more feature channels and halves image size after pooling.
        current_channels = in_channels
        for feature in features:
            self.downs.append(DoubleConv(current_channels, feature))
            current_channels = feature

        # Bottleneck: deepest part of the U-Net.
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        # Decoder: each step upsamples and combines with the matching encoder feature map.
        current_channels = features[-1] * 2
        for feature in reversed(features):
            self.ups.append(nn.ConvTranspose2d(current_channels, feature, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(feature * 2, feature))
            current_channels = feature

        # 1 x 1 convolution converts features into class scores for every pixel.
        self.final_conv = nn.Conv2d(features[0], num_classes, kernel_size=1)

    def forward(self, x):
        skip_connections = []

        # Encoder forward pass. Save skip connections before pooling.
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        # Decoder forward pass.
        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            skip_connection = skip_connections[idx // 2]

            # This handles rare shape mismatches caused by resizing or odd image sizes.
            if x.shape != skip_connection.shape:
                x = F.interpolate(x, size=skip_connection.shape[2:], mode='bilinear', align_corners=False)

            # Concatenate means join feature maps along the channel dimension.
            x = torch.cat((skip_connection, x), dim=1)
            x = self.ups[idx + 1](x)

        return self.final_conv(x)


model = UNet(in_channels=1, num_classes=NUM_CLASSES).to(DEVICE)

# Print a simple architecture summary for students.
print('U-Net encoder stages:', len(model.downs))
print('U-Net decoder stages:', len(model.ups) // 2)
print('Feature channels:', (32, 64, 128, 256))
print('Bottleneck channels:', 512)

# Quick shape test. This uses fake images to confirm the model input/output sizes.
with torch.no_grad():
    test_input = torch.randn(2, 1, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
    test_output = model(test_input)
print('Model output shape:', test_output.shape)


## 8. Loss function and metrics

The loss function tells the model how wrong it is during training. The optimizer uses this loss to update the model weights.

This notebook uses two losses together:

- **Cross-entropy loss**: teaches the model the correct class for each pixel.
- **Dice loss**: focuses on overlap and helps with class imbalance.

Class imbalance is important in medical segmentation. Most pixels may be background, while tumor pixels are much rarer. If we only care about total pixel accuracy, a model can look good while missing tumors.

Evaluation uses:

- **Dice score**: overlap between prediction and true mask. Higher is better. Best value is `1`.
- **IoU / Jaccard score**: intersection divided by union. Higher is better. Best value is `1`.


In [ ]:
def dice_loss(logits, targets, smooth=1e-6):
    """Multi-class Dice loss.

    logits:  batch x classes x height x width
    targets: batch x height x width with class numbers
    """
    # Convert raw model scores into probabilities between 0 and 1.
    probabilities = torch.softmax(logits, dim=1)

    # Convert class-number masks into one-hot masks.
    # Example: class 2 becomes [0, 0, 1].
    targets_one_hot = F.one_hot(targets, num_classes=NUM_CLASSES).permute(0, 3, 1, 2).float()

    # Sum over batch, height, and width. Keep one value per class.
    dims = (0, 2, 3)
    intersection = torch.sum(probabilities * targets_one_hot, dims)
    cardinality = torch.sum(probabilities + targets_one_hot, dims)
    dice_per_class = (2.0 * intersection + smooth) / (cardinality + smooth)

    # Exclude background from Dice loss so the model focuses more on liver/tumor.
    return 1.0 - dice_per_class[1:].mean()


def combined_loss(logits, targets):
    # Cross-entropy is strong for pixel classification.
    ce = F.cross_entropy(logits, targets)

    # Dice loss is useful when classes are imbalanced.
    dl = dice_loss(logits, targets)
    return ce + dl


def segmentation_metrics(logits, targets, num_classes=3, smooth=1e-6):
    """Return Dice and IoU for each class as a dictionary."""
    # argmax chooses the class channel with the highest score for each pixel.
    predictions = torch.argmax(logits, dim=1)
    metrics = {}

    for class_idx, class_name in enumerate(CLASS_NAMES[:num_classes]):
        pred_class = (predictions == class_idx)
        true_class = (targets == class_idx)

        intersection = (pred_class & true_class).sum().float()
        pred_sum = pred_class.sum().float()
        true_sum = true_class.sum().float()
        union = (pred_class | true_class).sum().float()

        dice = (2 * intersection + smooth) / (pred_sum + true_sum + smooth)
        iou = (intersection + smooth) / (union + smooth)

        metrics[f'dice_{class_name}'] = dice.item()
        metrics[f'iou_{class_name}'] = iou.item()

    return metrics


## 9. Training and validation functions

Training and validation are separate.

During training:

1. The model makes a prediction.
2. The loss compares the prediction with the true mask.
3. PyTorch calculates gradients.
4. The optimizer updates the model weights.

During validation:

1. The model makes predictions on validation images.
2. We calculate loss and metrics.
3. We do not update the model weights.

This separation helps us see whether the model is learning patterns that generalize beyond the training data.


In [ ]:
def train_one_epoch(model, loader, optimizer):
    """Train the model for one epoch and return the average loss."""
    model.train()  # turn on training behavior such as BatchNorm updates
    running_loss = 0.0

    for images, masks in tqdm(loader, desc='Training', leave=False):
        # Move tensors to GPU or CPU depending on DEVICE.
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        # Clear old gradients before calculating new gradients.
        optimizer.zero_grad()

        # Forward pass: model predicts class scores for every pixel.
        logits = model(images)

        # Compare prediction with true mask.
        loss = combined_loss(logits, masks)

        # Backward pass: calculate gradients.
        loss.backward()

        # Optimizer step: update model weights.
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):
    """Evaluate without updating model weights."""
    model.eval()  # turn on evaluation behavior
    running_loss = 0.0
    totals = {f'dice_{name}': 0.0 for name in CLASS_NAMES}
    totals.update({f'iou_{name}': 0.0 for name in CLASS_NAMES})
    n_batches = 0

    for images, masks in tqdm(loader, desc='Evaluating', leave=False):
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        logits = model(images)
        loss = combined_loss(logits, masks)
        batch_metrics = segmentation_metrics(logits, masks, num_classes=NUM_CLASSES)

        running_loss += loss.item() * images.size(0)
        for key, value in batch_metrics.items():
            totals[key] += value
        n_batches += 1

    results = {'loss': running_loss / len(loader.dataset)}
    for key, value in totals.items():
        results[key] = value / max(n_batches, 1)
    return results


## 10. Train the model

This cell runs the full training loop.

One epoch means the model has seen every training image once. After each epoch, the notebook evaluates the model on the validation set.

The notebook saves the best model according to validation tumor Dice. Tumor Dice is used because tumor segmentation is usually harder than liver segmentation and is often the most clinically interesting class.

If training is slow, reduce `IMAGE_SIZE`, `BATCH_SIZE`, or `EPOCHS` in the configuration cell.


In [ ]:
# Adam is a commonly used optimizer for deep learning.
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# history stores one row of results per epoch.
history = []
best_val_tumor_dice = -1.0
best_model_path = Path('best_unet_liver_tumor.pth')

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    # Train once over the full training set.
    train_loss = train_one_epoch(model, train_loader, optimizer)

    # Measure performance on the validation set.
    val_results = evaluate(model, val_loader)

    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        **{f'val_{k}': v for k, v in val_results.items()},
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {train_loss:.4f} | "
        f"val loss {val_results['loss']:.4f} | "
        f"val liver Dice {val_results['dice_liver']:.4f} | "
        f"val tumor Dice {val_results['dice_tumor']:.4f}"
    )

    # Save the model only when tumor Dice improves.
    if val_results['dice_tumor'] > best_val_tumor_dice:
        best_val_tumor_dice = val_results['dice_tumor']
        torch.save(model.state_dict(), best_model_path)
        print('  Saved new best model:', best_model_path)

print(f'Training time: {(time.time() - start_time) / 60:.1f} minutes')

# Convert history into a table for display and plotting.
history_df = pd.DataFrame(history)
history_df


## 11. Plot the training curves

Plots help us understand training behavior.

Typical patterns:

- Training loss should usually decrease.
- Validation loss should usually decrease at first.
- Dice scores should usually increase.
- If training improves but validation gets worse, the model may be overfitting.

Overfitting means the model memorizes training images instead of learning patterns that work on new images.


In [ ]:
if len(history) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train loss')
    axes[0].plot(history_df['epoch'], history_df['val_loss'], label='validation loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(history_df['epoch'], history_df['val_dice_liver'], label='liver Dice')
    axes[1].plot(history_df['epoch'], history_df['val_dice_tumor'], label='tumor Dice')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Dice score')
    axes[1].set_ylim(0, 1)
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()

## 12. Test-set evaluation

The test set should only be used after training choices are finished. It estimates how well the trained model performs on unseen data.

Do not repeatedly change settings based on the test results. If we tune the model using the test set, the test set is no longer a fair final evaluation.


In [ ]:
# Load the best saved model before testing.
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.to(DEVICE)

test_results = evaluate(model, test_loader)
test_df = pd.DataFrame([test_results]).T.rename(columns={0: 'test_value'})
test_df

## 13. Show model predictions

Numerical scores are useful, but visual inspection is also important.

The next cell compares:

- the CT image
- the true mask
- the predicted mask
- the prediction overlaid on the CT image

Look for common errors such as missing small tumors, liver boundary mistakes, or false tumor predictions.


In [ ]:
@torch.no_grad()
def predict_one(model, image_tensor):
    """Predict one segmentation mask from one image tensor."""
    model.eval()

    # The dataset returns one image with shape channels x height x width.
    # The model expects a batch, so unsqueeze(0) adds batch size 1.
    image_tensor = image_tensor.unsqueeze(0).to(DEVICE)

    logits = model(image_tensor)

    # Convert raw scores into class numbers by choosing the highest-scoring class.
    prediction = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()
    return prediction


def show_predictions(model, dataset, n=4):
    """Show image, true mask, predicted mask, and overlay."""
    indices = random.sample(range(len(dataset)), k=min(n, len(dataset)))
    fig, axes = plt.subplots(len(indices), 4, figsize=(16, 4 * len(indices)))
    if len(indices) == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, idx in enumerate(indices):
        image_tensor, true_mask = dataset[idx]
        pred_mask = predict_one(model, image_tensor)

        image_np = image_tensor.squeeze(0).numpy()
        true_mask_np = true_mask.numpy()

        axes[row, 0].imshow(image_np, cmap='gray')
        axes[row, 0].set_title('CT image')
        axes[row, 0].axis('off')

        axes[row, 1].imshow(class_mask_to_rgb(true_mask_np))
        axes[row, 1].set_title('True mask')
        axes[row, 1].axis('off')

        axes[row, 2].imshow(class_mask_to_rgb(pred_mask))
        axes[row, 2].set_title('Predicted mask')
        axes[row, 2].axis('off')

        axes[row, 3].imshow(image_np, cmap='gray')
        axes[row, 3].imshow(class_mask_to_rgb(pred_mask), alpha=0.35)
        axes[row, 3].set_title('Prediction overlay')
        axes[row, 3].axis('off')

    plt.tight_layout()
    plt.show()

show_predictions(model, test_dataset, n=4)


## 14. Save predicted masks for the test set

This creates PNG prediction masks in `predictions/test`.

The saved colors match the training masks:

- black = background
- green = liver
- red = tumor

These files can be opened like normal images or used for later analysis.


In [ ]:
@torch.no_grad()
def save_test_predictions(model, dataset, output_dir='predictions/test'):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    model.eval()

    for idx in tqdm(range(len(dataset)), desc='Saving predictions'):
        image_tensor, _ = dataset[idx]
        pred_mask = predict_one(model, image_tensor)
        pred_rgb = class_mask_to_rgb(pred_mask)

        source_name = dataset.image_paths[idx].name
        output_path = output_dir / source_name
        Image.fromarray(pred_rgb).save(output_path)

    return output_dir

prediction_dir = save_test_predictions(model, test_dataset)
print('Saved predictions to:', prediction_dir)

## 15. Grad-CAM explanation map

Grad-CAM means **Gradient-weighted Class Activation Mapping**. It is a method for visualizing which parts of an image strongly influence a neural network's decision.

For a normal image-classification model, Grad-CAM usually explains one class score, such as "cat" or "dog". Here we have a segmentation model, so the model gives a class score for every pixel.

In this notebook, Grad-CAM is used to explain the **tumor class**. The code asks: which regions of the CT image contribute most to the model's tumor score?

Important interpretation notes:

- Red/yellow areas in the heatmap are regions the model used more strongly.
- Blue/dark areas were used less strongly.
- Grad-CAM is not the same as the predicted segmentation mask.
- Grad-CAM is an explanation tool, not proof that the model is clinically correct.

The Grad-CAM code below uses the last encoder block, `model.downs[-1]`, because this layer contains high-level image features before the bottleneck.


In [ ]:
def make_gradcam(model, image_tensor, target_class=2):
    """Create a Grad-CAM heatmap for one image and one target class.

    target_class=2 means tumor because CLASS_NAMES[2] is 'tumor'.
    The returned heatmap has values between 0 and 1.
    """
    model.eval()

    # We use the last encoder block as the explanation layer.
    # It contains higher-level features than the early layers.
    target_layer = model.downs[-1]

    activations = []
    gradients = []

    def forward_hook(module, inputs, output):
        # Save feature maps produced by the target layer.
        activations.append(output)

    def backward_hook(module, grad_input, grad_output):
        # Save gradients flowing back from the target class score.
        gradients.append(grad_output[0])

    # Hooks let us look inside the model during forward and backward passes.
    forward_handle = target_layer.register_forward_hook(forward_hook)
    backward_handle = target_layer.register_full_backward_hook(backward_hook)

    try:
        # Add a batch dimension: channels x height x width becomes 1 x channels x height x width.
        image_batch = image_tensor.unsqueeze(0).to(DEVICE)

        # Forward pass: get segmentation logits.
        logits = model(image_batch)

        # For segmentation, there is a score for every pixel.
        # We take the mean tumor score over the whole output mask to get one value for Grad-CAM.
        class_score = logits[:, target_class, :, :].mean()

        # Backward pass: calculate gradients of that score with respect to the target layer.
        model.zero_grad()
        class_score.backward()

        activation = activations[0]
        gradient = gradients[0]

        # Average gradients over height and width to get one importance weight per feature channel.
        weights = gradient.mean(dim=(2, 3), keepdim=True)

        # Weighted sum of activation maps gives the class activation map.
        cam = (weights * activation).sum(dim=1, keepdim=True)
        cam = F.relu(cam)

        # Resize heatmap back to the input image size.
        cam = F.interpolate(cam, size=image_tensor.shape[1:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().detach().cpu().numpy()

        # Normalize to 0..1 for display.
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam

    finally:
        # Always remove hooks so they do not affect later model runs.
        forward_handle.remove()
        backward_handle.remove()


def show_gradcam_examples(model, dataset, n=3, target_class=2):
    """Display CT image, prediction, Grad-CAM heatmap, and overlay."""
    indices = random.sample(range(len(dataset)), k=min(n, len(dataset)))
    fig, axes = plt.subplots(len(indices), 4, figsize=(16, 4 * len(indices)))
    if len(indices) == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, idx in enumerate(indices):
        image_tensor, true_mask = dataset[idx]
        pred_mask = predict_one(model, image_tensor)
        cam = make_gradcam(model, image_tensor, target_class=target_class)

        image_np = image_tensor.squeeze(0).numpy()

        axes[row, 0].imshow(image_np, cmap='gray')
        axes[row, 0].set_title('CT image')
        axes[row, 0].axis('off')

        axes[row, 1].imshow(class_mask_to_rgb(pred_mask))
        axes[row, 1].set_title('Predicted mask')
        axes[row, 1].axis('off')

        axes[row, 2].imshow(cam, cmap='jet')
        axes[row, 2].set_title(f'Grad-CAM: {CLASS_NAMES[target_class]}')
        axes[row, 2].axis('off')

        axes[row, 3].imshow(image_np, cmap='gray')
        axes[row, 3].imshow(cam, cmap='jet', alpha=0.45)
        axes[row, 3].set_title('Grad-CAM overlay')
        axes[row, 3].axis('off')

    plt.tight_layout()
    plt.show()

# Show Grad-CAM explanations for the tumor class.
show_gradcam_examples(model, test_dataset, n=3, target_class=2)


## 16. Possible improvements:

- Train for more epochs.
- Use full `512 x 512` images if memory allows.
- Add more data augmentation.
- Use patient-level splitting if the current split was not already made by patient.
- Compare U-Net with another architecture.
- Compare Grad-CAM maps between correct and incorrect predictions.
